# OTTO · Round 05 — Build the action-pair index

One bounded stage at a time. Reuse existing pre-cutoff Parquet; no raw JSON scan, original graph rebuild, or new experiment-model fit. **Stop and return the ZIP on any pause or error.**

## 1 · Check the kernel
Select **OTTO - Notebook**, then run this cell alone. It should receive an execution number within 15 seconds.

In [1]:
from pathlib import Path
import importlib.util, json, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'transition_features.py').is_file() and (p/'launch.py').is_file()),
            Path.home()/'otto_feature_round05')
if not (ROOT/'launch.py').is_file():
    raise FileNotFoundError('Open the notebook from the extracted otto_feature_round05 folder.')
sys.path.insert(0,str(ROOT))
spec = importlib.util.spec_from_file_location('launch', ROOT/'launch.py')
launch = importlib.util.module_from_spec(spec)
sys.modules['launch'] = launch
spec.loader.exec_module(launch)
STATE={'halted':False, 'completed':[]}
def stage(name):
    if STATE['halted']:
        raise RuntimeError('An earlier stage stopped. Return the ZIP before running another stage.')
    try:
        value=launch.run_stage(name)
        STATE['completed'].append(name)
        return value
    except BaseException:
        STATE['halted']=True
        raise
print('KERNEL_READY', flush=True)
print('NOTEBOOK_PYTHON:',sys.executable)
print('ML_PYTHON_UNCHANGED:',Path.home()/'otto-recommender-system/.venv/bin/python')
print('PACKAGE:',ROOT)

KERNEL_READY
NOTEBOOK_PYTHON: /opt/conda/bin/python
ML_PYTHON_UNCHANGED: /home/sagemaker-user/otto-recommender-system/.venv/bin/python
PACKAGE: /home/sagemaker-user/otto_feature_round05


## 2 · Tests and installed-backend smoke
Run and wait for `ROUND05_TESTS_PASSED`. The tiny DuckDB/Parquet fixture is mandatory and uses the installed environment. No package installation.

In [2]:
stage('tests')

RUNNING tests; process cap 60s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round05/outputs/tests.log
test_disjoint_deterministic_selection (test_metric_contracts.CohortAndFoldTests.test_disjoint_deterministic_selection) ... ok
test_duplicate_fold_ids_rejected (test_metric_contracts.CohortAndFoldTests.test_duplicate_fold_ids_rejected) ... ok
test_forward_folds_full_replication_shape (test_metric_contracts.CohortAndFoldTests.test_forward_folds_full_replication_shape) ... ok
test_insufficient_embargo_support (test_metric_contracts.CohortAndFoldTests.test_insufficient_embargo_support) ... ok
test_invalid_cohort_input (test_metric_contracts.CohortAndFoldTests.test_invalid_cohort_input) ... ok
test_no_target_argument_in_selection (test_metric_contracts.CohortAndFoldTests.test_no_target_argument_in_selection) ... ok
test_seed_changes_membership (test_metric_contracts.CohortAndFoldTests.test_seed_changes_membership) ... ok
test_tied_queries_do_not_leak_across_time (test_me

{'phase': 'tests', 'exit_code': 0}

## 3 · Historical transition counts
**One invocation only.** Source/candidate requests are label-blind; all 5,120 study session IDs are excluded. Sixteen immutable source-ID partitions. 300-second useful-work limit, 320-second outer cap. `PAUSED_CHECKPOINTED` means return the report before any additional work, not automatically run again.

In [3]:
if 'tests' not in STATE['completed']:
    raise RuntimeError('Run and pass tests in this notebook first.')
stage('index')

RUNNING index; process cap 320s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round05/outputs/index.log
{"duckdb": "1.5.5", "event": "installed_backend_smoke_passed", "model_fits": 0, "status": "DUCKDB_TRANSITION_SMOKE_PASSED", "synthetic_events": 23, "utc": "2026-09-12T03:57:07.116135+00:00"}
{"bucket": 0, "contract_id": "463b5558efcc9b0d720309c3985dd2d95c7694591de63b11b40aa2e63b39eb59", "elapsed_seconds": 4.3547518989980745, "event": "transition_partition_committed", "path": "part-00.npz", "request_pairs": 222164, "sha256": "68b058a34f284d8192dfb0283d65284661932fa9635f0e69fb974b743d408934", "sources": 12216, "supported_request_pairs": 26686, "typed_support_units_including_nonrequested_targets": 2183604, "utc": "2026-09-12T03:57:20.302260+00:00"}
2026-09-12T03:57:21.666227+00:00 LAUNCHER_HEARTBEAT phase=index seconds=15.0
{"completed": 1, "elapsed_seconds": 15.0, "event": "heartbeat", "stage": "transition_partition", "total": 16, "utc": "2026-09-12T03:57:21.735560+0

{'phase': 'index', 'exit_code': 0}

## 4 · Review and save
Continue only after the complete index passes. Save this notebook before bundling.

In [4]:
summary=json.loads((ROOT/'outputs/index_summary.json').read_text())
assert summary['status']=='ROUND05_INDEX_READY'
assert len(summary['partitions'])==16
assert summary['study_sessions_excluded']==5120
print('ROUND05_INDEX_READY — partitions:',len(summary['partitions']))
print('Requested directed pairs:',summary['request_pairs'])
print('Next: 02_run_action_pair_comparison.ipynb')
launch.collect()

ROUND05_INDEX_READY — partitions: 16
Requested directed pairs: 3309444
Next: 02_run_action_pair_comparison.ipynb
RETURN_FILE: /home/sagemaker-user/otto_feature_round05/otto_round05_return.zip


'/home/sagemaker-user/otto_feature_round05/otto_round05_return.zip'